In [1]:
import torch
from pyqcu import tools, dslash, lattice
from pyqcu.cuda import qcu, define
from pyqcu.cuda.define import params, argv, set_ptrs
params[define._LAT_X_] = 4
params[define._LAT_Y_] = 4
params[define._LAT_Z_] = 4
params[define._LAT_T_] = 8
params[define._LAT_XYZT_] = params[define._LAT_X_] * \
    params[define._LAT_Y_]*params[define._LAT_Z_]*params[define._LAT_T_]
params[define._GRID_X_], params[define._GRID_Y_], params[define._GRID_Z_], params[
    define._GRID_T_] = tools.give_grid_size()
params[define._PARITY_] = 0
params[define._NODE_RANK_] = define.rank
params[define._NODE_SIZE_] = define.size
params[define._DAGGER_] = 0
params[define._MAX_ITER_] = 1000
params[define._DATA_TYPE_] = define._LAT_C64_
# params[define._DATA_TYPE_] = define._LAT_C128_
params[define._SET_INDEX_] = 0
params[define._SET_PLAN_] = 1
params[define._MG_X_] = 1
params[define._MG_Y_] = 1
params[define._MG_Z_] = 1
params[define._MG_T_] = 1
params[define._LAT_E_] = 24
params[define._VERBOSE_] = 1
params[define._SEED_] = 42
argv = argv.to(dtype=define.dtype(params[define._DATA_TYPE_]).to_real())
argv[define._MASS_] = 0.05
argv[define._TOL_] = 1e-9
argv[define._SIGMA_] = 0.1
print(params)
print(argv)
print(set_ptrs)
gauge_eo = torch.zeros(size=[2, 3, 3, 4]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                       params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))
fermion_in_eo = torch.zeros(size=[2, 4, 3]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                                            params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))
fermion_in_eo = torch.rand_like(fermion_in_eo)
fermion_out_eo = torch.zeros(size=[2, 4, 3]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                             params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))
params[define._VERBOSE_] = 1
params[define._SET_INDEX_] = 0
params[define._SET_PLAN_] = -1
params[define._PARITY_] = 0
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyGaussGaugeQcu(gauge_eo, set_ptrs, params)
print(lattice.check_su3(U=gauge_eo[0]))
print(lattice.check_su3(U=gauge_eo[1]))
qcu.applyEndQcu(set_ptrs, params)
print(set_ptrs)
params[define._VERBOSE_] = 1
params[define._SET_INDEX_] += 1
params[define._SET_PLAN_] = 1
params[define._PARITY_] = 0
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyWilsonBistabCgQcu(
    fermion_out_eo, fermion_in_eo, gauge_eo, set_ptrs, params)
qcu.applyEndQcu(set_ptrs, params)
print(set_ptrs)
qcu_dest = tools.poooxyzt2oooxyzt(input_array=fermion_out_eo)
qcu_U = tools.poooxyzt2oooxyzt(input_array=gauge_eo)
qcu_src = tools.poooxyzt2oooxyzt(input_array=fermion_in_eo)
refer_src = dslash.give_wilson(
    src=qcu_dest, U=qcu_U, kappa=1 / (2 * argv[define._MASS_] + 8), with_I=True)
print('qcu_src:', qcu_src.flatten()[:100])
print('refer_src:', refer_src.flatten()[:100])
print('Difference:', tools.norm(refer_src-qcu_src)/tools.norm(qcu_src))
clover_ee = torch.zeros(size=[4, 3, 4, 3]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                                           params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))
clover_ee_inv = torch.zeros(size=[4, 3, 4, 3]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                                               params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))
clover_oo = torch.zeros(size=[4, 3, 4, 3]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                                           params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))
clover_oo_inv = torch.zeros(size=[4, 3, 4, 3]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                                               params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))
# gauge_eo = torch.ones_like(gauge_eo)*2
params[define._VERBOSE_] = 1
params[define._SET_INDEX_] += 1
params[define._SET_PLAN_] = 2
params[define._PARITY_] = 0
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyCloversQcu(clover_ee, clover_ee_inv, gauge_eo, set_ptrs, params)
qcu.applyEndQcu(set_ptrs, params)
print(set_ptrs)
params[define._VERBOSE_] = 1
params[define._SET_INDEX_] += 1
params[define._SET_PLAN_] = 2
params[define._PARITY_] = 1
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyCloversQcu(clover_oo, clover_oo_inv, gauge_eo, set_ptrs, params)
qcu.applyEndQcu(set_ptrs, params)
print(set_ptrs)
fermion_out_eo = torch.zeros(size=[2, 4, 3]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                             params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))
params[define._VERBOSE_] = 1
params[define._SET_INDEX_] += 1
params[define._SET_PLAN_] = 1
params[define._PARITY_] = 0
qcu.applyInitQcu(set_ptrs, params, argv)
qcu.applyCloverBistabCgQcu(fermion_out_eo, fermion_in_eo, gauge_eo,
                           clover_ee, clover_oo, clover_ee_inv, clover_oo_inv,  set_ptrs, params)
qcu.applyEndQcu(set_ptrs, params)
print(set_ptrs)
qcu_dest = tools.poooxyzt2oooxyzt(input_array=fermion_out_eo)
qcu_U = tools.poooxyzt2oooxyzt(input_array=gauge_eo)
qcu_src = tools.poooxyzt2oooxyzt(input_array=fermion_in_eo)
refer_clover_term = dslash.make_clover(
    U=qcu_U, kappa=1 / (2 * argv[define._MASS_] + 8))
refer_src = dslash.give_wilson(
    src=qcu_dest, U=qcu_U, kappa=1 / (2 * argv[define._MASS_] + 8), with_I=True)+dslash.give_clover(src=qcu_dest, clover_term=refer_clover_term)
print('qcu_src:', qcu_src.flatten()[:100])
print('refer_src:', refer_src.flatten()[:100])
print('Difference:', tools.norm(refer_src-qcu_src)/tools.norm(qcu_src))
clover_eeoo = torch.zeros(size=[2, 4, 3, 4, 3]+[params[define._LAT_X_], params[define._LAT_Y_], params[define._LAT_Z_],
                                                params[define._LAT_T_]//define._LAT_P_]).to(dtype=define.dtype(params[define._DATA_TYPE_]), device=torch.device('cuda'))
clover_eeoo[0] = clover_ee
clover_eeoo[1] = clover_oo
qcu_clover_term = tools.poooxyzt2oooxyzt(
    input_array=clover_eeoo)
qcu_clover_term = dslash.cut_I(clover_term=qcu_clover_term)

qcu_clover_term_eo = tools.oooxyzt2poooxyzt(
    input_array=qcu_clover_term)
refer_clover_term_eo = tools.oooxyzt2poooxyzt(
    input_array=refer_clover_term)
print('qcu_clover_term:', qcu_clover_term.flatten()[:100])
print('refer_clover_term:', refer_clover_term.flatten()[:100])
# print('qcu_clover_term_eo:', qcu_clover_term_eo[0].flatten()[:100])
# print('refer_clover_term_eo:', refer_clover_term_eo[0].flatten()[:100])
# print('qcu_clover_term_eo:', qcu_clover_term_eo[1].flatten()[:100])
# print('refer_clover_term_eo:', refer_clover_term_eo[1].flatten()[:100])
print('Difference:', tools.norm(refer_clover_term -
      qcu_clover_term)/tools.norm(qcu_clover_term))


/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


tensor([   4,    4,    4,    8,  512,    1,    1,    1,    1,    0,    0,    1,
           0, 1000,    2,    0,    1,    1,    1,    1,    1,   24,    1,   42,
           0], dtype=torch.int32)
tensor([5.0000e-02, 1.0000e-09, 1.0000e-01])
tensor([0, 0, 0, 0, 0, 0, 0, 0, 0, 0])
set_ptr:0x55ab6ce2a6d0
set_ptrs:0x55ab6bad5d00
long long set_ptr:94194754561744
gridDim.x                    :16
blockDim.x                   :16
host_params[_LAT_X_]         :4
host_params[_LAT_Y_]         :4
host_params[_LAT_Z_]         :4
host_params[_LAT_T_]         :4
host_params[_LAT_XYZT_]      :256
host_params[_GRID_X_]        :1
host_params[_GRID_Y_]        :1
host_params[_GRID_Z_]        :1
host_params[_GRID_T_]        :1
host_params[_PARITY_]        :0
host_params[_NODE_RANK_]     :0
host_params[_NODE_SIZE_]     :1
host_params[_DAGGER_]        :0
host_params[_MAX_ITER_]      :1000
host_params[_DATA_TYPE_]     :2
host_params[_SET_INDEX_]     :0
host_params[_SET_PLAN_]      :-1
host_params[_MG_X_]       

In [4]:
print('qcu_dest:', qcu_dest.flatten()[:100])

qcu_dest: tensor([-771.0482-1199.7980j, -934.1281-1248.2379j, -837.5353-1125.1191j,
        -886.0417-1201.2151j, -818.8695-1158.1412j, -907.5605-1322.4980j,
        -820.7415-1136.3214j, -938.5887-1245.3716j, -838.9213-1258.0303j,
        -806.2278-1081.7562j, -903.5760-1141.1838j, -746.6223-1091.8479j,
        -866.2154-1245.7802j, -817.0255-1133.8778j, -912.1517-1236.8093j,
        -724.9016-1096.0347j, -762.4581-1118.6112j, -889.9106-1210.6598j,
        -731.6737-1069.6362j, -850.5221-1204.9241j, -780.9292-1055.1613j,
        -797.7758-1210.8062j, -748.5833-1118.4371j, -878.2435-1141.9950j,
        -858.2347-1171.0802j, -766.9030-1133.2708j, -894.2609-1191.5663j,
        -755.3085-1094.5599j, -935.5658-1197.9003j, -773.3178-1139.8947j,
        -933.2077-1170.6479j, -770.2507-1086.1750j, -931.6043-1222.8669j,
        -921.3709-1087.7736j, -918.1093-1135.8082j, -837.3054-1105.3907j,
        -959.9849-1159.2913j, -913.5742-1104.0439j, -922.5110-1149.8007j,
        -873.6030-1076.0825j

In [3]:
a = qcu_src-refer_src
print('a:', a.flatten()[:100])

a: tensor([-11.2483-16.1176j, 249.1160+166.1341j, -11.4814-15.4619j,
        252.1882+184.9812j, -11.7482-16.2753j, 236.2161+191.8456j,
         -9.9372-16.6813j, 236.5872+174.0800j, 202.2105+158.3033j,
        -11.2821-15.1821j, 209.0818+142.9494j, -10.1305-17.7665j,
        221.4930+156.3300j, -11.5493-16.4068j, 204.8202+146.5064j,
         -9.5715-16.2052j,  -9.8134-16.9162j, 213.7388+155.7942j,
        -11.4617-17.5186j, 205.9909+162.4195j, -10.5078-16.9383j,
        209.3504+155.0135j, -10.0736-16.2683j, 195.7794+128.0247j,
        189.5466+122.2012j, -11.4053-14.7675j, 201.8636+137.7841j,
        -10.5750-17.0444j, 207.7327+146.4029j,  -9.6893-16.7690j,
        209.1851+142.1898j, -10.8471-17.1173j,  87.3999+117.6742j,
          5.9895-4.6502j, 100.0159+140.4634j,   4.9752-5.3293j,
        100.2084+118.6904j,   5.0643-5.9391j,  96.0687+124.9976j,
          5.6121-5.4187j,   5.2731-5.5329j,  51.5252+86.5107j,
          5.3806-6.3715j,  51.5048+96.7604j,   5.1067-5.9050j,
         